# Libraries

In [ ]:
# Import libraries
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/sumn2u/garbage-classification-v2"

Creating train, test and validation data

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(128,128),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)

Building a model

In [ ]:
#model = tf.keras.Sequential([
    
#    tf.keras.layers.Conv2D(32,(3,3),activation="relu",input_shape=(128,128,3)),
#    tf.keras.layers.MaxPooling2D(),

 #   tf.keras.layers.Conv2D(64,(3,3),activation="relu"),
 #   tf.keras.layers.MaxPooling2D(),

#    tf.keras.layers.Conv2D(128,(3,3),activation="relu"),
 #   tf.keras.layers.MaxPooling2D(),

  #  tf.keras.layers.Flatten(),

   # tf.keras.layers.Dense(128,activation="relu"),
    #tf.keras.layers.Dense(train_data.num_classes,activation="softmax")
#])

Compile

In [ ]:
#model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

Train

In [ ]:
#history = mcodel.fit(
 #   train_data,
 #   validation_data=val_data,
 #  epochs=3
#)

### Data set size was huge, reducing number of images

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

Dataset

In [ ]:
DATASET_PATH = Path("/kaggle/input/datasets/sumn2u/garbage-classification-v2")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

In [ ]:
from pathlib import Path
import random

DATASET_PATH = Path("/kaggle/input/datasets/sumn2u/garbage-classification-v2")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

original = [x for x in (DATASET_PATH / "original").rglob("*") 
            if x.is_file() and x.suffix.lower() in IMG_EXTS]

standardized_256 = [x for x in (DATASET_PATH / "standardized_256").rglob("*") 
                    if x.is_file() and x.suffix.lower() in IMG_EXTS]

standardized_384 = [x for x in (DATASET_PATH / "standardized_384").rglob("*") 
                    if x.is_file() and x.suffix.lower() in IMG_EXTS]

print("original:", len(original))
print("standardized_256:", len(standardized_256))
print("standardized_384:", len(standardized_384))

In [ ]:
from PIL import Image

def keep_valid_images(file_list):
    valid_files = []
    for f in file_list:
        try:
            with Image.open(f) as img:
                img.verify()
            valid_files.append(f)
        except:
            pass
    return valid_files

original = keep_valid_images(original)
standardized_256 = keep_valid_images(standardized_256)
standardized_384 = keep_valid_images(standardized_384)

print("After validation:")
print("original:", len(original))
print("standardized_256:", len(standardized_256))
print("standardized_384:", len(standardized_384))

In [ ]:
random.seed(42)

original = random.sample(original, min(len(original), 8000))
standardized_256 = random.sample(standardized_256, min(len(standardized_256), 4000))
standardized_384 = random.sample(standardized_384, min(len(standardized_384), 4000))

In [ ]:
files = []
labels = []

for f in original:
    files.append(str(f))
    labels.append(0)

for f in standardized_256 + standardized_384 :
    files.append(str(f))
    labels.append(1)

print("Total images:", len(files))

In [ ]:
from sklearn.model_selection import train_test_split

train_files, test_files, train_labels, test_labels = train_test_split(
    files, labels, test_size=0.15, random_state=42, stratify=labels
)

train_files, val_files, train_labels, val_labels = train_test_split(
    train_files, train_labels, test_size=0.15, random_state=42, stratify=train_labels
)

print(len(train_files), len(val_files), len(test_files))

In [ ]:
IMG_SIZE = (224, 224)

def preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

In [ ]:
 def create_dataset(files, labels):
    
        ds = tf.data.Dataset.from_tensor_slices((files, labels))
    
        ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.apply(tf.data.experimental.ignore_errors())  # ADD THIS LINE
    
        ds = ds.batch(32)
    
        ds = ds.prefetch(tf.data.AUTOTUNE)
    
        return ds

In [ ]:
train_ds = create_dataset(train_files, train_labels)
val_ds = create_dataset(val_files, val_labels)
test_ds = create_dataset(test_files, test_labels)

In [ ]:
plt.figure(figsize=(8,8))

for images, labels in train_ds.take(1):

    for i in range(9):

        plt.subplot(3,3,i+1)

        plt.imshow(images[i])

        plt.title("original" if labels[i]==1 else "standardized")

        plt.axis("off")

plt.show()

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
model.compile(

    optimizer="adam",

    loss="binary_crossentropy",

    metrics=["accuracy"]
)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        verbose=1
    )
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

test_loss, test_accuracy = model.evaluate(test_ds)
print("Test Accuracy:", test_accuracy)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Accuracy:", test_accuracy)

# Quantum CNN

In [ ]:
# Install PennyLane
!pip install pennylane pennylane-lightning -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml
import numpy as np

############################################
# Convert TensorFlow Dataset → NumPy
############################################

def tf_to_numpy(dataset):

    images = []
    labels = []

    for x, y in dataset:
        images.append(x.numpy())
        labels.append(y.numpy())

    images = np.concatenate(images)
    labels = np.concatenate(labels)

    return images, labels


X_train, y_train = tf_to_numpy(train_ds)
X_test, y_test   = tf_to_numpy(test_ds)

############################################
# Convert NumPy → PyTorch
############################################

X_train = torch.tensor(X_train).permute(0,3,1,2).float()
X_test  = torch.tensor(X_test).permute(0,3,1,2).float()

y_train = torch.tensor(y_train).float()
y_test  = torch.tensor(y_test).float()

############################################
# DataLoader
############################################

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
testloader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

############################################
# Quantum Circuit
############################################

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")

def quantum_circuit(inputs, weights):

    qml.AngleEmbedding(inputs, wires=range(n_qubits))

    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


weight_shapes = {"weights": (3, n_qubits, 3)}

quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

############################################
# Hybrid Quantum CNN
############################################

class HybridQCNN(nn.Module):

    def __init__(self):

        super(HybridQCNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((7,7))
        )

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(8 * 7 * 7, 4)

        self.quantum = quantum_layer

        self.fc2 = nn.Linear(4, 1)

    def forward(self, x):

        x = self.conv(x)

        x = self.flatten(x)

        x = self.fc1(x)

        x = torch.tanh(x)

        x = self.quantum(x)

        x = self.fc2(x)

        return torch.sigmoid(x)


model = HybridQCNN()

############################################
# Training Setup
############################################

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

############################################
# Training
############################################

for epoch in range(5):

    running_loss = 0.0

    for images, labels in trainloader:

        labels = labels.unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}")


############################################
# Testing
############################################

correct = 0
total = 0

with torch.no_grad():

    for images, labels in testloader:

        labels = labels.unsqueeze(1)

        outputs = model(images)

        predicted = (outputs > 0.5).float()

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Draw with matplotlib
fig, ax = qml.draw_mpl(quantum_circuit)(sample_inputs, sample_weights)
plt.show()

# Save the figure
fig.savefig('quantum_circuit.png', dpi=300, bbox_inches='tight')

In [ ]:
print(qml.draw(quantum_circuit)(torch.randn(4), torch.randn(3,4,3)))

# improvements without sir suggestion

In [ ]:
# Install PennyLane
!pip install pennylane pennylane-lightning -q

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml

In [ ]:
############################################
# Convert TensorFlow Dataset -> NumPy -> PyTorch
############################################

def tf_to_numpy(dataset):
    images = []
    labels = []

    for x, y in dataset:
        images.append(x.numpy())
        labels.append(y.numpy())

    images = np.concatenate(images, axis=0)
    labels = np.concatenate(labels, axis=0)

    return images, labels


X_train, y_train = tf_to_numpy(train_ds)
X_val, y_val     = tf_to_numpy(val_ds)
X_test, y_test   = tf_to_numpy(test_ds)

# NHWC -> NCHW for PyTorch
X_train = torch.tensor(X_train).permute(0, 3, 1, 2).float()
X_val   = torch.tensor(X_val).permute(0, 3, 1, 2).float()
X_test  = torch.tensor(X_test).permute(0, 3, 1, 2).float()

# 10-class labels
y_train = torch.tensor(y_train).long()
y_val   = torch.tensor(y_val).long()
y_test  = torch.tensor(y_test).long()

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valloader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
testloader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
############################################
# Quantum Circuit + Hybrid QCNN
############################################

n_qubits = 6
n_q_layers = 2
n_classes = 10

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_q_layers, n_qubits, 3)}
quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)


class HybridQCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.flatten = nn.Flatten()

        self.fc1 = nn.Sequential(
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_qubits)
        )

        self.quantum = quantum_layer

        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = torch.tanh(x)
        x = self.quantum(x)
        x = self.classifier(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HybridQCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

# Optional circuit drawing
sample_inputs = torch.tensor(np.random.rand(n_qubits), dtype=torch.float32)
sample_weights = torch.tensor(np.random.rand(n_q_layers, n_qubits, 3), dtype=torch.float32)

fig, ax = qml.draw_mpl(quantum_circuit)(sample_inputs, sample_weights)
plt.show()
fig.savefig("quantum_circuit.png", dpi=300, bbox_inches="tight")

In [ ]:
############################################
# Training + Validation + Testing
############################################

best_val_acc = 0.0
best_state = None
epochs = 25

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

    train_acc = 100 * train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict()

    print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

# Load best model
model.load_state_dict(best_state)

# Test accuracy
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        test_total += labels.size(0)
        test_correct += (preds == labels).sum().item()

test_acc = 100 * test_correct / test_total
print(f"\nBest Validation Accuracy: {best_val_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%")

## now thw qcnn code with sir suggestion

In [ ]:
!pip install pennylane pennylane-lightning -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml
import numpy as np

In [ ]:
def tf_to_numpy(dataset):
    images, labels = [], []

    for x, y in dataset:
        images.append(x.numpy())
        labels.append(y.numpy())

    images = np.concatenate(images, axis=0) / 255.0
    labels = np.concatenate(labels, axis=0)

    return images, labels


X_train, y_train = tf_to_numpy(train_ds)
X_test, y_test   = tf_to_numpy(test_ds)

X_train = torch.tensor(X_train).permute(0,3,1,2).float()
X_test  = torch.tensor(X_test).permute(0,3,1,2).float()

y_train = torch.tensor(y_train).float().view(-1,1)
y_test  = torch.tensor(y_test).float().view(-1,1)

trainloader = DataLoader(TensorDataset(X_train,y_train), batch_size=16, shuffle=True)
testloader  = DataLoader(TensorDataset(X_test,y_test), batch_size=16)

In [ ]:
n_qubits = 6
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    # Encode input
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")

    # Variational block (acts like dynamic + convolution)
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


weight_shapes = {"weights": (2, n_qubits, 3)}
quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)


class HybridQCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,8,3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((7,7))
        )

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(8*7*7, 6)

        # Feature importance
        self.alpha = nn.Parameter(torch.ones(6))

        self.quantum = quantum_layer

        # Residual + quantum
        self.fc2 = nn.Linear(12,1)

    def forward(self,x):
        x = self.conv(x)
        x = self.flatten(x)
        x = self.fc1(x)

        # residual
        x_res = x.clone()

        # feature importance encoding
        x = torch.tanh(self.alpha * x)

        # ⚡ fast quantum (no loop)
        q_out = self.quantum(x)

        # residual fusion
        x = torch.cat([q_out, x_res], dim=1)

        x = self.fc2(x)

        return torch.sigmoid(x)


model = HybridQCNN().float()
print("Fast model ready 🚀")

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x,y in trainloader:
        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out,y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred = (out>0.5).float()
        correct += (pred==y).sum().item()
        total += y.size(0)

    train_acc = 100*correct/total

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x,y in testloader:
            out = model(x)
            pred = (out>0.5).float()
            correct += (pred==y).sum().item()
            total += y.size(0)

    test_acc = 100*correct/total

    print(f"Epoch {epoch+1} | Loss {total_loss:.4f} | Train {train_acc:.2f}% | Test {test_acc:.2f}%")

# 